相较v2的改进点：

对特征之间的相关性检验，综合筛选出合适的特征

In [24]:
import pandas as pd
from sklearn.linear_model import LogisticRegression


train = pd.read_csv('/Kaggle-competition/Diabetes/train_diabetes.csv')
test = pd.read_csv('/Kaggle-competition/Diabetes/train_diabetes.csv')
train

,PatientID,Age,Gender,BMI,Glucose,BloodPressure,SkinThickness,Insulin,HbA1c,ExerciseHours,DietScore,Smoking,AlcoholConsumption,FamilyHistory,Diabetes
0,1,58,1,20.6,110,83,35.0,85.0,4.9,0.6,10,0,1,0,0
1,2,66,1,26.3,102,63,20.0,127.0,4.1,0.2,4,1,1,0,0
2,3,79,0,29.5,111,75,NaN,154.0,4.7,1.7,6,0,0,1,1
3,4,76,1,28.5,70,61,33.0,88.0,4.0,2.3,9,0,0,0,1
4,5,30,0,29.1,128,70,NaN,140.0,6.7,1.2,7,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,796,34,1,18.8,124,79,27.0,NaN,6.6,5.4,2,1,0,0,1
796,797,36,1,18.0,92,79,NaN,193.0,4.7,0.1,7,1,2,0,1
797,798,23,0,20.4,99,82,41.0,NaN,4.7,8.0,4,0,1,0,0
798,799,54,1,26.8,82,86,NaN,180.0,5.4,4.4,7,0,0,0,1


In [14]:
#从train数据中取出所有特征列（去掉目标变量Diabetes），并用中位数填充缺失值
features = train.drop('Diabetes',axis=1).fillna(train.median())
#计算相关性矩阵
corr_matrix = features.corr().abs()

# 只看相关性最高的5对特征
print("相关性最高的5对特征:")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        high_corr.append((
            corr_matrix.columns[i],
            corr_matrix.columns[j],
            corr_matrix.iloc[i, j]
        ))


#按相关性排序，取前5
for feat1, feat2, corr in sorted(high_corr, key=lambda x: x[2], reverse=True)[:5]:
    print(f"{feat1:15} 和 {feat2:15}: {corr:.3f}")


相关性最高的5对特征:
Gender          和 Insulin        : 0.079
HbA1c           和 Smoking        : 0.077
Smoking         和 FamilyHistory  : 0.075
ExerciseHours   和 Smoking        : 0.074
Gender          和 BloodPressure  : 0.071


通过特征之间的相关性检验，选择Gender,HbA1c,FamilyHistory

In [19]:
train['Gender'].isna().sum()
train['HbA1c'].isna().sum()
train['FamilyHistory'].isna().sum()

test['Gender'].isna().sum()
test['HbA1c'].isna().sum()
test['FamilyHistory'].isna().sum()
# 检查后，无缺失值

np.int64(0)

In [22]:
features = ['Gender', 'HbA1c', 'FamilyHistory']
X_train = train[features]
y_train = train['Diabetes']
X_test = test[features]

model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

output = pd.DataFrame({'PatientID':test['PatientID'],'Diabetes': predictions})
output.to_csv('/Users/caierchang/Desktop/Diabetes_prediction_v3.csv', index=False)

版本3预测结果详细评估

基本表现指标

准确率：68.50% - 200个样本中预测对了137个

精确率：73.74% - 当你预测为糖尿病时，正确率73.74%

召回率：63.00% - 你找到了63%的实际糖尿病患者